# 智慧潍苑扫码 → 超星登录 → 房间/时段 → 选座预览
本 notebook 使用本机代理，沿用官方请求流程。**没有真实预约功能**：客户端的 submit 方法和网络发送层都阻止预约提交；只开放已确认的查询接口。当前支持本校普通模式、服务器配置时段、网格或列表座位。未知模式明确停止。

新内核首次运行需要你在手机端扫码确认。如果当前内核已经有 `login`，默认复用，不重新扫码。

In [ ]:
import importlib
from IPython.display import Image, display
import qr_login, seat_guard, seat_client
importlib.reload(seat_guard)
importlib.reload(qr_login)
importlib.reload(seat_client)

PROXY = "http://127.0.0.1:7890"
FID_ENC = "35bbd135397006a8"
MAPP_ID = "4109435"
ROOM_ID = 6299
DAY = None  # None 使用服务器当天；也可填 YYYY-MM-DD

new_scan = "login" not in globals()
if new_scan:
    login = qr_login.QRLogin(proxy=PROXY)
    display(Image(data=login.start(), format="png"))
else:
    seat_guard.guard_session(login.session)
    qr_login.configure_login_session(login.session)
    print("复用当前内核的登录会话。")

若上面显示二维码，先运行下面的等待，再扫码并在手机上确认。若登录已过期，重新创建 `login = qr_login.QRLogin(proxy=PROXY)`、调用 `login.start()` 展示新二维码并等待。不要重复使用旧授权码。

In [ ]:
if new_scan:
    login_result = login.wait_for_scan(seconds=120)
    print(login_result)
else:
    print("跳过重复扫码，下一步检查会话。")

### 进入座位首页
无需粘贴旧 `wfw_token`。先用 Cookie 会话验证首页，再查询房间。

In [ ]:
client = seat_client.SeatClient(login, FID_ENC, MAPP_ID)
print(client.open_home())
rooms = client.list_rooms(DAY)
for room in rooms:
    print(room)

### 选择房间和时段
修改前面的 ROOM_ID 后，从本格重新运行。只展示尚未开始且未暂停的时段，比网页允许预约当前时段的规则更保守。

In [ ]:
room = client.open_room(ROOM_ID)
print({k:v for k,v in room.items() if k != "selectable_time_slots"})
slots = room["selectable_time_slots"]
for i, slot in enumerate(slots):
    print(i, slot)

In [ ]:
SLOT_INDEX = 0  # 按上一格输出修改
if not slots:
    raise ValueError("当天没有未来开放时段，请更换日期或房间")
START_TIME = slots[SLOT_INDEX]["startTime"]
END_TIME = slots[SLOT_INDEX]["endTime"]
available = client.available_seats(START_TIME, END_TIME)
print(f"{START_TIME}–{END_TIME}，当前空闲 {len(available)} 个座位")
print(available)

### 选择座位（仅本地状态）
修改 SEAT_NUM 选择其他空闲座位。默认优先 196 号；若占用，选择返回的第一个空闲座位。选座前会再次查询占用情况。

In [ ]:
if not available:
    raise ValueError("该时段无空闲座位")
SEAT_NUM = "196" if "196" in available else available[0]
selection = client.choose(SEAT_NUM, START_TIME, END_TIME)
print(selection.summary())

现在已完成选座和签名准备，**没有预约成功、没有占用座位**。`selection.prepared_form()` 仅生成表单文本，不联网；不要把 Cookie 或签名输入写进共享日志。验证码/风控提交分支不在此只读流程中。

不要使用其他未加拦截的 HTTP 客户端测试提交。此处 `client.submit()` 始终抛出 `BlockedSeatRequest`。诊断日志位于 `work/login_logs/`，不纳入 Git。